In [ ]:
import copy
from scipy.optimize import minimize
import scipy
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator
from canari import (
    DataProcess,
    Model,
    plot_states,
    plot_data,
    plot_prediction,
)
from canari.component import (
    Exponential,
    WhiteNoise,
    LocalTrend,
    Periodic,
    LocalLevel,
    Autoregression,
    LocalAcceleration,
)

os.makedirs("scipy_test_AB15G100_kalman_acc_finalv2", exist_ok=True)
df_raw = pd.read_csv(
    "/Users/michelwu/Desktop/Exp DAT/reel_data/AB15G100.DAT",
    sep=";",  # Semicolon as delimiter
    quotechar='"',  # Double quotes as text qualifier
    engine="python",  # Python engine for complex cases
    na_values=[""],  # Treat empty strings as NaN
    skipinitialspace=True,  # Skip spaces after delimiter
    encoding="ISO-8859-1",
    parse_dates=["Date"],
    index_col="Date",
)

def log_likelihood_validation(y_true, mu_pred, std_pred):
    """Log-vraisemblance gaussienne."""
    y_true = y_true.flatten()
    mu_pred = mu_pred.flatten()
    std_pred = std_pred.flatten()

    mask = ~np.isnan(y_true)
    y_true = y_true[mask]
    mu_pred = mu_pred[mask]
    std_pred = std_pred[mask]

    ll = -0.5 * np.sum(
        np.log(2 * np.pi * std_pred**2) + ((y_true - mu_pred) ** 2) / (std_pred**2)
    )
    return ll


def mse_general(y_true, mu_pred):
    y_true = y_true.flatten()
    mu_pred = mu_pred.flatten()
    mask = ~np.isnan(y_true)
    return np.mean((y_true[mask] - mu_pred[mask]) ** 2)


def mae_general(y_true, mu_pred):
    y_true = y_true.flatten()
    mu_pred = mu_pred.flatten()
    mask = ~np.isnan(y_true)
    return np.mean(np.abs(y_true[mask] - mu_pred[mask]))

df = -df_raw[["Deplacements cumulatif X (mm)"]]
df = df.drop(pd.to_datetime("2010-08-24 08:00:00"))
df = df.iloc[:]
mask = ~np.isnan(df)
df = df[mask].resample("M").mean()
# print(df)
first_year = df.index.min().year
last_year = df.index.max().year
years = [str(year) for year in range(first_year, last_year - 4)]
validation_start_str = df.loc[str(last_year - 4)].index[0].strftime("%Y-%m-%d")
test_start_str = df.loc[str(last_year - 4)].index[0].strftime("%Y-%m-%d")
last_date_str=df.index[-1].strftime("%Y-%m-%d")
# print(df.loc[years[0]].index[0].strftime("%Y-%m-%d"))
data_processor = DataProcess(
    data=df,
    train_start=df.loc[years[0]].index[0].strftime("%Y-%m-%d"),
    train_end=validation_start_str,
    validation_start=validation_start_str,
    validation_end=last_date_str,
    test_start=last_date_str,
    output_col=[0],
    standardization=False,
)
train_data, validation_data, test_data, all_data = data_processor.get_splits()

print(len(train_data["y"]))
print(len(validation_data["y"]))
print(len(test_data["y"]))
print(len(all_data["y"]))
df_train=pd.DataFrame(index=train_data["time"], data={'y':train_data["y"].flatten()})

df.index.name="date"

In [ ]:
expo=Exponential()
local_level=LocalLevel()
ar=Autoregression(phi=0,mu_states=[0,0,0,0])
wn=WhiteNoise()
periodic=Periodic(period=12)
model = Model(expo,ar,local_level,periodic)
model.auto_initialize_comp(data_training=df_train,ratio_training=0.6)
# print(model.mu_states)
# print(model.var_states)
# print(model.process_noise_matrix)
model.mu_states[model.get_states_index("level")]=-1.75
model.var_states[model.get_states_index("level"),model.get_states_index("level")]=(-1.75*0.02)**2

model.filter(data=train_data)
model.smoother(matrix_inversion_tol=1e-12)
# model.filter(data=validation_data)
mu_test,std_test,states=model.forecast(data=validation_data)

ll_expo = log_likelihood_validation(validation_data["y"], mu_test, std_test)
mse_expo = mse_general(validation_data["y"], mu_test)
mae_expo = mae_general(validation_data["y"], mu_test)

print(ll_expo,mse_expo,mae_expo)

In [ ]:
scaled_exp_index = model.get_states_index("scaled exp")
level_index = model.get_states_index("level")
periodic_index = model.get_states_index("periodic 1")
autoregression_index = model.get_states_index("autoregression")
cov_scaled_exp_level = []
cov_scaled_exp_periodic = []
cov_level_periodic = []
cov_level_auto = []
cov_scaled_exp_auto = []
cov_periodic_auto = []
for i in range(len(model.states.get_mean("level",states_type="smooth"))):
    cov_scaled_exp_level.append(
        model.states.var_smooth[i][scaled_exp_index, level_index]
    )
    cov_scaled_exp_periodic.append(
        model.states.var_smooth[i][scaled_exp_index, periodic_index]
    )
    cov_level_periodic.append(model.states.var_smooth[i][level_index, periodic_index])
    cov_level_auto.append(
        model.states.var_smooth[i][
            level_index, model.get_states_index("autoregression")
        ]
    )
    cov_scaled_exp_auto.append(
        model.states.var_smooth[i][
            scaled_exp_index, model.get_states_index("autoregression")
        ]
    )
    cov_periodic_auto.append(
        model.states.var_smooth[i][
            periodic_index, model.get_states_index("autoregression")
        ]
    )
cov_scaled_exp_level = np.array(cov_scaled_exp_level)
cov_scaled_exp_periodic = np.array(cov_scaled_exp_periodic)
cov_level_periodic = np.array(cov_level_periodic)
cov_level_auto = np.array(cov_level_auto)
cov_scaled_exp_auto = np.array(cov_scaled_exp_auto)
cov_periodic_auto = np.array(cov_periodic_auto)

# Plot scaled exp add to local level
# print(len(df.index))
# print(len(model.states.get_mean("scaled exp", states_type="smooth")))
all_dates = data_processor.data.index
n_train = len(train_data["y"])
n_validation = len(validation_data["y"])
# 1 = len(test_data["y"])
# 1 = 0
train_dates = all_dates[: (n_train + n_validation)]
all_y_values = data_processor.get_data("all")


my_width = 2.5
my_height = 1.6
print(len(all_dates))

# plt.figure(figsize=(my_width, my_height))


# Calculer la moyenne et l'écart-type
y_mean_expo = (
    model.states.get_mean("scaled exp", states_type="smooth")
    + model.states.get_mean("level", states_type="smooth")
    + model.states.get_mean("periodic 1", states_type="smooth")
    + model.states.get_mean("autoregression", states_type="smooth")
)
y_std_expo = np.sqrt(
    model.states.get_std("scaled exp", states_type="smooth") ** 2
    + model.states.get_std("level", states_type="smooth") ** 2
    + model.states.get_std("periodic 1", states_type="smooth") ** 2
    + model.states.get_std("autoregression", states_type="smooth") ** 2
    + 2
    * (
        cov_scaled_exp_level
        + cov_scaled_exp_periodic
        + cov_level_periodic
        + cov_level_auto
        + cov_periodic_auto
        + cov_scaled_exp_auto
    )
)
y_exp_level = model.states.get_mean("scaled exp", states_type="smooth") + model.states.get_mean(
    "level", states_type="smooth"
)
print(len(y_mean_expo))

# --- CORRECTION ICI ---
# Tracer la moyenne lissée avec les VRAIES DATES
plt.plot(
    all_dates[len(all_dates)-1 - n_train - n_validation : len(all_dates)-1],
    y_mean_expo,
    color="purple",
)  # <--- PAS np.arange

plt.plot(
    all_dates[len(all_dates)-1 - n_train - n_validation : len(all_dates)-1],
    y_exp_level,
    color="blue",
)
# Tracer l'intervalle de confiance avec les VRAIES DATES
plt.fill_between(
    all_dates[
        len(all_dates) - n_train - n_validation-1 : len(all_dates)-1
    ],  # <--- PAS np.arange
    y_mean_expo + y_std_expo,
    y_mean_expo - y_std_expo,
    color="purple",
    alpha=0.3,
    edgecolor="none",
)

# Tracer toutes les données brutes avec les VRAIES DATES
plt.scatter(
    all_dates[
        len(all_dates) - n_train - n_validation-1  : len(all_dates)-1
    ],  # <--- PAS np.arange
    all_y_values[
        len(all_dates) - n_train - n_validation-1  : len(all_dates)-1
    ],  # <--- Y correspondant
    color="red",
    s=2.5,
)
plt.axvline(
    x=pd.to_datetime(f"{last_year-4}"),
    color="green",  # Ou la couleur de votre choix
    linestyle="--",  # Style en tirets
    linewidth=1,
)
plt.axvspan(
    xmin=pd.to_datetime(f"{last_year-4}"),
    xmax=pd.to_datetime(f"{last_year}"),
    color="green",  # Couleur de base
    alpha=0.2,  # Transparence pour obtenir un "vert clair",
)
# --- FIN DE LA CORRECTION ---

ax = plt.gca()
ax.set_yticks([])
# Formate l'étiquette pour n'afficher que l'année
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
# Met une étiquette tous les 3 ans
ax.xaxis.set_major_locator(mdates.YearLocator(base=3))
# Force les limites min/max de l'axe
ax.set_xlim(pd.to_datetime(f"{first_year}"), pd.to_datetime(f"{last_year}"))
# --- Fin de la modification ---

plt.title("Modèle exponentiel")
plt.xlabel("Année")
plt.ylabel("Déplacement")
plt.show()

In [ ]:
local_acceleration=LocalAcceleration()
model_acc=Model(local_acceleration,periodic,ar)
model_acc.auto_initialize_comp(data_training=df_train,ratio_training=0.6)
print(model_acc.auto_initialize_comp(data_training=df_train,ratio_training=0.6))
model_acc.mu_states[model_acc.get_states_index("level")]=-1.75
model_acc.var_states[model_acc.get_states_index("level"),model_acc.get_states_index("level")]=(-1.75*0.02)**2
model_acc.filter(data=train_data)
model_acc.smoother(matrix_inversion_tol=1e-12)
# model.filter(data=validation_data)
mu_test_acc,std_test_acc,states_acc=model_acc.forecast(data=validation_data)

ll_acc = log_likelihood_validation(validation_data["y"], mu_test_acc, std_test_acc)
mse_acc = mse_general(validation_data["y"], mu_test_acc)
mae_acc = mae_general(validation_data["y"], mu_test_acc)

print(ll_acc,mse_acc,mae_acc)

In [ ]:
level_acc_index = model_acc.get_states_index("level")
periodic_acc_index = model_acc.get_states_index("periodic 1")
cov_level_acc_periodic = []
cov_level_acc_auto = []
# + cov_scaled_exp_auto
cov_periodic_acc_auto = []
for i in range(len(model_acc.states.get_mean("level", states_type="smooth"))):
    cov_level_acc_periodic.append(
        model_acc.states.var_smooth[i][level_acc_index, periodic_acc_index]
    )
    cov_level_acc_auto.append(
        model_acc.states.var_smooth[i][
            level_acc_index, model_acc.get_states_index("autoregression")
        ]
    )
    cov_periodic_acc_auto.append(
        model_acc.states.var_smooth[i][
            periodic_acc_index, model_acc.get_states_index("autoregression")
        ]
    )

cov_level_acc_periodic = np.array(cov_level_acc_periodic)
cov_level_acc_auto = np.array(cov_level_acc_auto)
cov_periodic_acc_auto = np.array(cov_periodic_acc_auto)

In [ ]:
plt.plot(
    all_dates[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    model_acc.states.get_mean("level", states_type="smooth")
    + model_acc.states.get_mean("periodic 1", states_type="smooth")
    + model_acc.states.get_mean("autoregression", states_type="smooth"),
    color="purple",
)
plt.plot(
    all_dates[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    model_acc.states.get_mean("level", states_type="smooth"),
    color="blue",
)
eps = 1e-10
plt.fill_between(
    all_dates[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    model_acc.states.get_mean("level", states_type="smooth")
    + model_acc.states.get_mean("periodic 1", states_type="smooth")
    + model_acc.states.get_mean("autoregression", states_type="smooth")
    + np.sqrt(
        model_acc.states.get_std("level", states_type="smooth") ** 2
        + model_acc.states.get_std("periodic 1", states_type="smooth") ** 2
        + model_acc.states.get_std("autoregression", states_type="smooth") ** 2
        + 2
        * (
            cov_level_acc_periodic
            + cov_level_acc_auto
            # + cov_scaled_exp_auto
            + cov_periodic_acc_auto
        )
        +eps
    ),
    model_acc.states.get_mean("level", states_type="smooth")
    + model_acc.states.get_mean("periodic 1", states_type="smooth")
    + model_acc.states.get_mean("autoregression", states_type="smooth")
    - np.sqrt(
        model_acc.states.get_std("level", states_type="smooth") ** 2
        + model_acc.states.get_std("periodic 1", states_type="smooth") ** 2
        + model_acc.states.get_std("autoregression", states_type="smooth") ** 2
        + 2
        * (
            cov_level_acc_periodic
            + cov_level_acc_auto
            # + cov_scaled_exp_auto
            + cov_periodic_acc_auto
        )
        +eps
    ),
    color="purple",
    alpha=0.3,
    edgecolor="none",
)
plt.scatter(
    all_dates[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    all_y_values[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    color="red",
    s=2.5,
)
ax = plt.gca()
# Formate l'étiquette pour n'afficher que l'année
ax.set_yticks([])
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
# Met une étiquette tous les 3 ans
ax.xaxis.set_major_locator(mdates.YearLocator(base=3))
# Force les limites min/max de l'axe
ax.set_xlim(pd.to_datetime(f"{first_year}"), pd.to_datetime(f"{last_year}"))
# --- Fin de la modification ---
plt.axvline(
    x=pd.to_datetime(f"{last_year-4}"),
    color="green",  # Ou la couleur de votre choix
    linestyle="--",  # Style en tirets
    linewidth=1,
)
plt.axvspan(
    xmin=pd.to_datetime(f"{last_year-4}"),
    xmax=pd.to_datetime(f"{last_year}"),
    color="green",  # Couleur de base
    alpha=0.2,  # Transparence pour obtenir un "vert clair"
)

plt.title("Modèle accélération")
plt.xlabel("Année")
plt.ylabel("Déplacement")

plt.show()

In [ ]:
local_trend=LocalTrend()
model_trend=Model(local_trend,periodic,ar)
model_trend.auto_initialize_comp(data_training=df_train,ratio_training=0.6)
print(model_trend.auto_initialize_comp(data_training=df_train,ratio_training=0.6))
model_trend.mu_states[model_trend.get_states_index("level")]=-1.75
model_trend.var_states[model_trend.get_states_index("level"),model_trend.get_states_index("level")]=(-1.75*0.02)**2
model_trend.filter(data=train_data)
model_trend.smoother(matrix_inversion_tol=1e-12)
# model.filter(data=validation_data)
mu_test_trend,std_test_trend,states_trend=model_trend.forecast(data=validation_data)

ll_trend = log_likelihood_validation(validation_data["y"], mu_test_trend, std_test_trend)
mse_trend = mse_general(validation_data["y"], mu_test_trend)
mae_trend = mae_general(validation_data["y"], mu_test_trend)

print(ll_trend,mse_trend,mae_trend)

In [ ]:

level_trend_index = model_trend.get_states_index("level")
periodic_trend_index = model_trend.get_states_index("periodic 1")
cov_level_trend_periodic = []
cov_level_trend_auto = []
# + cov_scaled_exp_auto
cov_periodic_trend_auto = []
for i in range(len(model_trend.states.get_mean("level", states_type="smooth"))):
    cov_level_trend_periodic.append(
        model_trend.states.var_smooth[i][level_trend_index, periodic_trend_index]
    )
    cov_level_trend_auto.append(
        model_trend.states.var_smooth[i][
            level_trend_index, model_trend.get_states_index("autoregression")
        ]
    )
    cov_periodic_trend_auto.append(
        model_trend.states.var_smooth[i][
            periodic_trend_index, model_trend.get_states_index("autoregression")
        ]
    )

cov_level_trend_periodic = np.array(cov_level_trend_periodic)
cov_level_trend_auto = np.array(cov_level_trend_auto)
cov_periodic_trend_auto = np.array(cov_periodic_trend_auto)
plt.plot(
    all_dates[len(all_dates) - n_train - n_validation -1 : len(all_dates)-1],
    model_trend.states.get_mean("level", states_type="smooth")
    + model_trend.states.get_mean("periodic 1", states_type="smooth")
    + model_trend.states.get_mean("autoregression", states_type="smooth"),
    color="purple",
)
plt.plot(
    all_dates[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    model_trend.states.get_mean("level", states_type="smooth"),
    color="blue",
)
# print(
#     model_trend.states.get_std("level", states_type="smooth") ** 2
#     + model_trend.states.get_std("periodic 1", states_type="smooth") ** 2
#     + model_trend.states.get_std("autoregression", states_type="smooth") ** 2
#     + 2
#     * (
#         cov_level_trend_periodic
#         + cov_level_trend_auto
#         # + cov_scaled_exp_auto
#         + cov_periodic_trend_auto
#     )
# )
plt.fill_between(
    all_dates[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    model_trend.states.get_mean("level", states_type="smooth")
    + model_trend.states.get_mean("periodic 1", states_type="smooth")
    + model_trend.states.get_mean("autoregression", states_type="smooth")
    + np.sqrt(
        model_trend.states.get_std("level", states_type="smooth") ** 2
        + model_trend.states.get_std("periodic 1", states_type="smooth") ** 2
        + model_trend.states.get_std("autoregression", states_type="smooth") ** 2
        + 2
        * (
            cov_level_trend_periodic
            + cov_level_trend_auto
            # + cov_scaled_exp_auto
            + cov_periodic_trend_auto
        )
        + eps
    ),
    model_trend.states.get_mean("level", states_type="smooth")
    + model_trend.states.get_mean("periodic 1", states_type="smooth")
    + model_trend.states.get_mean("autoregression", states_type="smooth")
    - np.sqrt(
        model_trend.states.get_std("level", states_type="smooth") ** 2
        + model_trend.states.get_std("periodic 1", states_type="smooth") ** 2
        + model_trend.states.get_std("autoregression", states_type="smooth") ** 2
        + 2
        * (
            cov_level_trend_periodic
            + cov_level_trend_auto
            # + cov_scaled_exp_auto
            + cov_periodic_trend_auto
        )
        + eps
    ),
    color="purple",
    alpha=0.3,
    edgecolor="none",
)
plt.scatter(
    all_dates[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    all_y_values[len(all_dates) - n_train - n_validation - 1 : len(all_dates)-1],
    color="red",
    s=2.5,
)
ax = plt.gca()
# Formate l'étiquette pour n'afficher que l'année
ax.set_yticks([])
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
# Met une étiquette tous les 3 ans
ax.xaxis.set_major_locator(mdates.YearLocator(base=3))
# Force les limites min/max de l'axe
ax.set_xlim(pd.to_datetime(f"{first_year}"), pd.to_datetime(f"{last_year}"))
# --- Fin de la modification ---
plt.axvline(
    x=pd.to_datetime(f"{last_year-4}"),
    color="green",  # Ou la couleur de votre choix
    linestyle="--",  # Style en tirets
    linewidth=1,
)
plt.axvspan(
    xmin=pd.to_datetime(f"{last_year-4}"),
    xmax=pd.to_datetime(f"{last_year}"),
    color="green",  # Couleur de base
    alpha=0.2,  # Transparence pour obtenir un "vert clair"
)

plt.title("Modèle trend")
plt.xlabel("Année")
plt.ylabel("Déplacement")
plt.show()

In [ ]:
plt.rcParams.update(
    {
        "text.usetex": True,  # Utiliser le moteur LaTeX pour générer le texte
        "font.family": "serif",  # Utiliser une police avec empattement
        "font.serif": ["Computer Modern"],  # Forcer Computer Modern
        "font.size": 10,  # Correspond au '1000' de ecrm1000 (10pt)
        # ASTUCE CRUCIALE POUR LE FRANÇAIS (puisque vous avez 'ecrm')
        # Cela force Python à charger le pack d'accents comme votre document
        "text.latex.preamble": r"\usepackage[T1]{fontenc} \usepackage[utf8]{inputenc}",
        # Ajustement des tailles pour être cohérent avec un document
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
    }
)
fig, axes = plt.subplots(1, 3, figsize=(7.5, 1.3), sharey=True)

# Configuration de la police (C'est parfait, on garde)
# Définition des indices de dates pour le plot
slice_idx = slice(len(all_dates) - n_train - n_validation - 1, len(all_dates)-1)
dates_plot = all_dates[slice_idx]
y_true_plot = all_y_values[slice_idx]

# =============================================================================
# 1. GRAPHE EXPO (axes[0])
# =============================================================================
# Recalcul des vecteurs pour être sûr (copie de ta logique Expo)
y_mean_expo = (
    model.states.get_mean("scaled exp", states_type="smooth")
    + model.states.get_mean("level", states_type="smooth")
    + model.states.get_mean("periodic 1", states_type="smooth")
    + model.states.get_mean("autoregression", states_type="smooth")
)
# Note: J'utilise ta logique de covariance spécifique pour l'Expo
y_std_expo = np.sqrt(
    model.states.get_std("scaled exp", states_type="smooth") ** 2
    + model.states.get_std("level", states_type="smooth") ** 2
    + model.states.get_std("periodic 1", states_type="smooth") ** 2
    + model.states.get_std("autoregression", states_type="smooth") ** 2
    + 2
    * (
        cov_scaled_exp_level
        + cov_scaled_exp_periodic
        + cov_level_periodic
        + cov_level_auto
        + cov_periodic_auto
        + cov_scaled_exp_auto
    )
)
y_level_expo = model.states.get_mean("scaled exp", states_type="smooth") + model.states.get_mean(
    "level", states_type="smooth"
)
LINE_WIDTH = 0.6
ax = axes[0]
ax.plot(
    dates_plot, y_mean_expo, color="purple", label="Prédiction", linewidth=LINE_WIDTH
)
ax.plot(dates_plot, y_level_expo, color="blue", label="Niveau", linewidth=LINE_WIDTH)
ax.fill_between(
    dates_plot,
    y_mean_expo + y_std_expo,
    y_mean_expo - y_std_expo,
    color="purple",
    alpha=0.3,
    edgecolor="none",
)
ax.scatter(dates_plot, y_true_plot, color="red", s=1, label="Données")
# ax.set_title("Exponential Model", fontsize=8)
ax.set_ylabel("Dataset 3", fontsize=8)  # Seul le 1er a le label Y

# =============================================================================
# 2. GRAPHE TREND (axes[1])
# =============================================================================
# Recalcul des vecteurs pour Trend
y_mean_trend = (
    model_trend.states.get_mean("level", states_type="smooth")
    + model_trend.states.get_mean("periodic 1", states_type="smooth")
    + model_trend.states.get_mean("autoregression", states_type="smooth")
)
y_std_trend = np.sqrt(
    model_trend.states.get_std("level", states_type="smooth") ** 2
    + model_trend.states.get_std("periodic 1", states_type="smooth") ** 2
    + model_trend.states.get_std("autoregression", states_type="smooth") ** 2
    + 2 * (cov_level_trend_periodic + cov_level_trend_auto + cov_periodic_trend_auto)
    + eps
)
y_level_trend = model_trend.states.get_mean("level", states_type="smooth")

ax = axes[1]
ax.plot(dates_plot, y_mean_trend, color="purple", linewidth=LINE_WIDTH)
ax.plot(dates_plot, y_level_trend, color="blue", linewidth=LINE_WIDTH)
ax.fill_between(
    dates_plot,
    y_mean_trend + y_std_trend,
    y_mean_trend - y_std_trend,
    color="purple",
    alpha=0.3,
    edgecolor="none",
)
ax.scatter(dates_plot, y_true_plot, color="red", s=1)
# ax.set_title("Local Trend Model", fontsize=8)

# =============================================================================
# 3. GRAPHE ACCELERATION (axes[2])
# =============================================================================
# Recalcul des vecteurs pour Accélération
y_mean_acc = (
    model_acc.states.get_mean("level", states_type="smooth")
    + model_acc.states.get_mean("periodic 1", states_type="smooth")
    + model_acc.states.get_mean("autoregression", states_type="smooth")
)
y_std_acc = np.sqrt(
    model_acc.states.get_std("level", states_type="smooth") ** 2
    + model_acc.states.get_std("periodic 1", states_type="smooth") ** 2
    + model_acc.states.get_std("autoregression", states_type="smooth") ** 2
    + 2 * (cov_level_acc_periodic + cov_level_acc_auto + cov_periodic_acc_auto)
    + eps
)
y_level_acc = model_acc.states.get_mean("level", states_type="smooth")

ax = axes[2]
ax.plot(dates_plot, y_mean_acc, color="purple", linewidth=LINE_WIDTH)
ax.plot(dates_plot, y_level_acc, color="blue", linewidth=LINE_WIDTH)
ax.fill_between(
    dates_plot,
    y_mean_acc + y_std_acc,
    y_mean_acc - y_std_acc,
    color="purple",
    alpha=0.3,
    edgecolor="none",
)
ax.scatter(dates_plot, y_true_plot, color="red", s=1)
# ax.set_title("Acceleration Model", fontsize=8)

# =============================================================================
# FORMATAGE COMMUN (Dates, Lignes verticales, etc.)
# =============================================================================
for ax in axes:
    # Ligne verticale et zone de test
    ax.axvline(
        x=pd.to_datetime(f"{last_year-4}"), color="green", linestyle="--", linewidth=1
    )
    ax.axvspan(
        xmin=pd.to_datetime(f"{last_year-4}"),
        xmax=pd.to_datetime(f"{last_year}"),
        color="green",
        alpha=0.2,
    )

    # Formatage Axe X
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(
        mdates.YearLocator(base=4)
    )  # Espacement ajusté pour éviter le chevauchement
    ax.set_xlim(pd.to_datetime(f"{first_year}"), pd.to_datetime(f"{last_year}"))
    # ax.set_xticks([])
    ax.set_yticks([])
# Label X seulement si nécessaire (souvent redondant avec les dates)
# On travaille sur le premier axe
ax0 = axes[0]

# 1. Récupérer les limites VISUELLES exactes de l'axe X (gauche et droite de l'image)
# Ce sont des nombres flottants (Matplotlib dates)
x_min_visuel, x_max_visuel = ax0.get_xlim()

# 2. Récupérer la position exacte de la séparation (ligne verte)
# On convertit la date de coupure en "nombre matplotlib" pour le calcul
split_date_num = mdates.date2num(pd.to_datetime(f"{last_year-4}"))

# 3. Calcul des milieux géométriques
# Milieu Zone Training = Moyenne entre le bord gauche et la ligne verte
mid_train_num = x_min_visuel + (split_date_num - x_min_visuel) * 0.8

# Milieu Zone Test = Moyenne entre la ligne verte et le bord droit
mid_test_num = split_date_num + (x_max_visuel - split_date_num) / 2

# 4. Affichage
text_props = dict(
    transform=ax0.get_xaxis_transform(),  # X en data (dates), Y en relatif (0-1)
    y=0.85,  # Hauteur (85% du graphique)
    ha="center",  # Centrage Horizontal (C'est ça qui compte !)
    va="center",  # Centrage Vertical
    fontsize=9,
)

# On reconvertit les nombres en dates pour la fonction text (si nécessaire selon version mpl,
# mais souvent text accepte les nombres si l'axe est date. Par sécurité on convertit).
# ax0.text(x=mdates.num2date(mid_train_num), s=r"Training", **text_props)
# ax0.text(x=mdates.num2date(mid_test_num), s=r"Test", **text_props)
plt.subplots_adjust(
    left=0.10,  # 10% de marge gauche (Place pour "fontsize=9acement Sensor X")
    right=0.99,  # 1% de marge à droite
    bottom=0.15,  # 15% en bas (Place pour les dates si présentes)
    top=0.85,  # 15% en haut (Place pour les titres "Exponential Model" etc)
    wspace=0.05,  # Espace entre les 3 colonnes
)
plt.savefig(
    "modele_comparaison/comparaison_modelesAB15G100.pdf",
    bbox_inches="tight",
)
# Sauvegarde
plt.show()
plt.close()


In [ ]:
print(mse_expo,mae_expo,ll_expo)
print(mse_acc,mae_acc,ll_acc)
print(mse_trend,mae_trend,ll_trend)